# Aura Fishing - Thai Transcription (Whisper large-v3)
1. Runtime > Change runtime type > **T4 GPU**
2. Run cell 1 (install)
3. Run cell 2 and upload `audio.mp3`
4. Run cell 3; `transcript_raw.txt` and `transcript_raw.srt` download automatically when it finishes

In [ ]:
!pip -q install faster-whisper

In [ ]:
from google.colab import files
up = files.upload()
AUDIO = list(up.keys())[0]
print(AUDIO)

In [ ]:
from faster_whisper import WhisperModel
from google.colab import files

model = WhisperModel('large-v3', device='cuda', compute_type='float16')
segments, info = model.transcribe(AUDIO, language='th', beam_size=5, vad_filter=True,
                                  condition_on_previous_text=False)

def ts(s):
    h, r = divmod(int(s), 3600); m, sec = divmod(r, 60)
    return f'{h:02d}:{m:02d}:{sec:02d}'
def srt_ts(s):
    ms = int((s - int(s)) * 1000)
    return ts(s) + f',{ms:03d}'

with open('transcript_raw.txt', 'w', encoding='utf-8') as t, open('transcript_raw.srt', 'w', encoding='utf-8') as srt:
    for i, seg in enumerate(segments, 1):
        line = f'[{ts(seg.start)} - {ts(seg.end)}] {seg.text.strip()}'
        t.write(line + '\n')
        srt.write(f'{i}\n{srt_ts(seg.start)} --> {srt_ts(seg.end)}\n{seg.text.strip()}\n\n')
        print(line)

files.download('transcript_raw.txt')
files.download('transcript_raw.srt')